In [2]:
# --- 1. Ferramentas que vamos usar ---
import pandas as pd            # pandas: trabalha com tabelas (DataFrames)
from pathlib import Path       # Path: ajuda a achar arquivos em pastas

In [3]:
# --- 2. Localizar os arquivos CSV de cada ano ---
# procura na pasta "dados" todo arquivo que começa com "datatran" e termina em .csv
# sorted() coloca em ordem: 2022, 2023, 2024...
arquivos = sorted(Path("dados").glob("datatran*.csv"))

In [4]:
# --- 3. Ler cada arquivo ---
# sep=";"            -> as colunas são separadas por ponto e vírgula
# encoding="latin-1" -> evita que acentos virem símbolos estranhos
# decimal=","        -> números usam vírgula decimal (ex: -7,29)
# resultado: uma lista com uma tabela por ano
partes = [pd.read_csv(f, sep=";", encoding="latin-1", decimal=",") for f in arquivos]

In [5]:
# --- 4. Conferir se todos os anos têm as mesmas colunas ---
colunas = list(partes[0].columns)          # colunas do primeiro arquivo (referência)
for f, p in zip(arquivos, partes):         # percorre cada arquivo junto com sua tabela
    # assert = alarme: se as colunas forem diferentes, o código para e avisa qual arquivo
    assert list(p.columns) == colunas, f"{f.name} tem colunas diferentes"

In [6]:
# --- 5. Juntar todos os anos numa tabela só ---
# ignore_index=True renumera as linhas de 0 em diante
df = pd.concat(partes, ignore_index=True)

In [7]:
# --- 6. Ajustar os tipos das colunas ---
df["id"] = df["id"].astype("int64")                  # id como número inteiro (em 2026 vinha decimal)
df["data_inversa"] = pd.to_datetime(df["data_inversa"])  # texto -> data de verdade
df["ano"] = df["data_inversa"].dt.year               # cria a coluna "ano" a partir da data

In [8]:
# --- 7. Verificação: quantos acidentes por ano? ---
# groupby("ano") separa por ano e size() conta as linhas de cada grupo
print(df.groupby("ano").size())

ano
2022    64606
2023    67766
2024    73202
2025    72529
2026    48712
dtype: int64


In [9]:
# --- 9. Limpeza: remover acidentes sem rodovia identificada ---
n_antes = len(df)
df = df[df["br"] > 0].copy()   # br=0 e km=0 são registros sem rodovia válida
print(f"Descartados (br=0): {n_antes - len(df)} de {n_antes} ({(n_antes - len(df)) / n_antes:.2%})")

Descartados (br=0): 815 de 326815 (0.25%)


In [10]:
# --- 10. Criar o trecho: rodovia + estado + faixa de 10 km ---
df["faixa_km"] = (df["km"] // 10 * 10).astype(int)
df["trecho"] = df["br"].astype(str) + "-" + df["uf"] + "-" + df["faixa_km"].astype(str)

In [11]:
# --- 11. Marcar acidente grave (morto ou ferido grave) ---
df["grave"] = ((df["mortos"] > 0) | (df["feridos_graves"] > 0)).astype(int)

In [12]:
# --- 12. Separar os dois períodos do projeto ---
# 2022-2024 definem quais trechos são críticos
# 2025-2026 testam se continuam críticos
df["periodo"] = df["ano"].map(lambda a: "definicao" if a <= 2024 else "teste")

print(df.groupby("periodo").size()) 

periodo
definicao    205026
teste        120974
dtype: int64


In [13]:
# --- 13. Top 5 causas mais graves ---
MIN_N = 500        # mínimo de acidentes por causa (2022-2026 juntos)
MIN_N_ANO = 100    # mínimo de acidentes por causa em cada ano

def resumo_causas(d):
    r = (d.groupby("causa_acidente")
           .agg(acidentes=("id", "size"), graves=("grave", "sum"), mortos=("mortos", "sum"))
           .reset_index())
    r["pct_do_total"] = (r["acidentes"] / len(d) * 100).round(1)   # peso da causa no total
    r["pct_graves"] = (r["graves"] / r["acidentes"] * 100).round(1) # % dos acidentes da causa que foram graves
    return r

# Todos os anos juntos (2022-2026)
geral = resumo_causas(df)
top5_geral = (geral[geral["acidentes"] >= MIN_N]
              .sort_values("pct_graves", ascending=False).head(5).reset_index(drop=True))
print(top5_geral)

# Cada ano separado
blocos = []
for ano, d in df.groupby("ano"):
    r = resumo_causas(d)
    r = r[r["acidentes"] >= MIN_N_ANO].sort_values("pct_graves", ascending=False).head(5)
    r.insert(0, "ano", ano)
    blocos.append(r)
top5_ano = pd.concat(blocos, ignore_index=True)
print(top5_ano)

# Exportar para o Power BI
top5_geral.to_csv("top5_causas_geral.csv", index=False, sep=";", encoding="utf-8-sig")
top5_ano.to_csv("top5_causas_por_ano.csv", index=False, sep=";", encoding="utf-8-sig")

                           causa_acidente  acidentes  graves  mortos  \
0                    Suicídio (presumido)        550     453     296   
1                Pedestre andava na pista       2948    2240    1255   
2           Entrada inopinada do pedestre       3175    2161     937   
3  Pedestre cruzava a pista fora da faixa       2625    1741     668   
4                  Transitar na contramão      11018    6639    4176   

   pct_do_total  pct_graves  
0           0.2        82.4  
1           0.9        76.0  
2           1.0        68.1  
3           0.8        66.3  
4           3.4        60.3  
     ano                                     causa_acidente  acidentes  \
0   2022                           Pedestre andava na pista        662   
1   2022             Pedestre cruzava a pista fora da faixa        581   
2   2022                      Entrada inopinada do pedestre        714   
3   2022                             Transitar na contramão       2055   
4   2022  Área ur

In [14]:
# --- 14. Tabela por trecho ---

# Contagens por trecho e período (definição x teste)
base = (df.groupby(["trecho", "periodo"])
          .agg(n=("id", "size"), graves=("grave", "sum"), mortos=("mortos", "sum"))
          .unstack("periodo", fill_value=0))
base.columns = [f"{a}_{b}" for a, b in base.columns]

# Taxa de gravidade em % (fica vazia se o trecho não teve acidente no período)
base["taxa_definicao"] = (base["graves_definicao"] / base["n_definicao"] * 100).round(1)
base["taxa_teste"] = (base["graves_teste"] / base["n_teste"] * 100).round(1)

# Identificação do trecho e posição média (para o mapa do Power BI)
info = (df.groupby("trecho")
          .agg(br=("br", "first"), uf=("uf", "first"), faixa_km=("faixa_km", "first"),
               lat=("latitude", "median"), lon=("longitude", "median")))

# Causa e tipo de acidente mais frequentes no período de definição
causa = (df[df["periodo"] == "definicao"].groupby("trecho")["causa_acidente"]
           .agg(lambda s: s.mode().iat[0]).rename("causa_principal"))
tipo = (df[df["periodo"] == "definicao"].groupby("trecho")["tipo_acidente"]
          .agg(lambda s: s.mode().iat[0]).rename("tipo_principal"))

trechos = info.join(base).join(causa).join(tipo).reset_index()

# Distribuição de n: serve para escolher o mínimo de acidentes
print("Total de trechos:", len(trechos))
print(trechos["n_definicao"].describe().round(1))
for m in (10, 20, 30):
    print(f"Trechos com pelo menos {m} acidentes em 2022-2024:", (trechos["n_definicao"] >= m).sum())

# Exportar para o Power BI
trechos.to_csv("trechos.csv", index=False, sep=";", encoding="utf-8-sig")

Total de trechos: 6506
count    6506.0
mean       31.5
std        72.4
min         0.0
25%         4.0
50%        11.0
75%        29.0
max      1661.0
Name: n_definicao, dtype: float64
Trechos com pelo menos 10 acidentes em 2022-2024: 3519
Trechos com pelo menos 20 acidentes em 2022-2024: 2229
Trechos com pelo menos 30 acidentes em 2022-2024: 1594


In [15]:
# --- 15. Trechos críticos ---
from scipy import stats   # se der erro, rode no terminal: pip install scipy

MIN_N = 20   # mínimo de acidentes em 2022-2024

# Taxa esperada: a taxa nacional de gravidade no período de definição
p0 = df.loc[df["periodo"] == "definicao", "grave"].mean()
print(f"Taxa nacional de gravidade (2022-2024): {p0:.1%}")

trechos["elegivel"] = trechos["n_definicao"] >= MIN_N
el = trechos[trechos["elegivel"]].copy()

# p-valor: chance de ver tantos graves (ou mais) se o trecho fosse igual à média nacional
el["p_valor"] = stats.binom.sf(el["graves_definicao"] - 1, el["n_definicao"], p0)

# Correção para múltiplos testes (Benjamini-Hochberg), porque testamos milhares de trechos
el["q_valor"] = stats.false_discovery_control(el["p_valor"], method="bh")
el["critico"] = el["q_valor"] < 0.05

# Devolver o resultado para a tabela completa
trechos = trechos.drop(columns=["p_valor", "q_valor", "critico"], errors="ignore")
trechos = trechos.merge(el[["trecho", "p_valor", "q_valor", "critico"]], on="trecho", how="left")
trechos["critico"] = trechos["critico"].fillna(False).astype(bool)

print("Trechos elegíveis:", int(trechos["elegivel"].sum()))
print("Trechos críticos:", int(trechos["critico"].sum()))
print(trechos[trechos["critico"]]
      .sort_values("taxa_definicao", ascending=False)
      [["trecho", "n_definicao", "graves_definicao", "taxa_definicao", "q_valor", "causa_principal"]]
      .head(10))

trechos.to_csv("trechos.csv", index=False, sep=";", encoding="utf-8-sig")

Taxa nacional de gravidade (2022-2024): 28.4%
Trechos elegíveis: 2229
Trechos críticos: 132
          trecho  n_definicao  graves_definicao  taxa_definicao   q_valor  \
4079  316-MA-590           20                18            90.0  0.000033   
610   116-BA-340           21                16            76.2  0.001148   
2942  232-PE-210           24                18            75.0  0.000553   
5736   423-PE-30           20                15            75.0  0.001724   
4326  324-BA-430           21                15            71.4  0.003632   
4973  369-PR-420           24                17            70.8  0.001709   
3126  242-BA-860           24                17            70.8  0.001709   
5746   424-PE-10           22                15            68.2  0.006304   
13     10-MA-230           25                17            68.0  0.003186   
159   101-BA-420           28                19            67.9  0.001598   

                                        causa_principal  
40

In [16]:
# --- 16. Teste de persistência (2025-2026) ---
from scipy import stats
import numpy as np

MIN_N_TESTE = 11   # mínimo de acidentes no período de teste (proporcional a 20 meses)

p0_def = df.loc[df["periodo"] == "definicao", "grave"].mean()
p0_teste = df.loc[df["periodo"] == "teste", "grave"].mean()

# Trechos que podem ser avaliados: elegíveis e com acidentes suficientes no teste
av = trechos[trechos["elegivel"] & (trechos["n_teste"] >= MIN_N_TESTE)].copy()

# Mesmo teste da célula 15, agora com os dados de 2025-2026
av["p_teste"] = stats.binom.sf(av["graves_teste"] - 1, av["n_teste"], p0_teste)
av["q_teste"] = stats.false_discovery_control(av["p_teste"], method="bh")
av["critico_teste"] = av["q_teste"] < 0.05

crit = av[av["critico"]]
resto = av[~av["critico"]]

def taxa(d, g, n):
    return d[g].sum() / d[n].sum() * 100

print(f"Críticos: {int(trechos['critico'].sum())} | avaliáveis no teste: {len(crit)} | não avaliáveis: {int(trechos['critico'].sum()) - len(crit)}")
for nome, d in [("Críticos", crit), ("Não críticos", resto)]:
    print(f"{nome}: {taxa(d,'graves_definicao','n_definicao'):.1f}% (definição) -> {taxa(d,'graves_teste','n_teste'):.1f}% (teste)")
print(f"Média nacional: {p0_def*100:.1f}% -> {p0_teste*100:.1f}%")

excesso_def = taxa(crit, "graves_definicao", "n_definicao") - p0_def * 100
excesso_teste = taxa(crit, "graves_teste", "n_teste") - p0_teste * 100
print(f"Parte do excesso de gravidade que se manteve: {excesso_teste / excesso_def:.0%}")
print(f"Críticos com taxa acima da média nacional no teste: {(crit['taxa_teste'] > p0_teste * 100).mean():.0%}")
print(f"Críticos que continuam críticos no teste: {crit['critico_teste'].mean():.0%}")
print(f"Não críticos que viraram críticos no teste: {resto['critico_teste'].mean():.0%}")
rho, p = stats.spearmanr(av["taxa_definicao"], av["taxa_teste"])
print(f"Correlação de Spearman entre as taxas dos dois períodos: {rho:.2f}")

# Status de cada trecho crítico, para o dashboard
trechos = trechos.drop(columns=["p_teste", "q_teste", "critico_teste", "status_teste"], errors="ignore")
trechos = trechos.merge(av[["trecho", "p_teste", "q_teste", "critico_teste"]], on="trecho", how="left")
trechos["status_teste"] = np.select(
    [~trechos["critico"],
     trechos["critico_teste"].isna(),
     trechos["critico_teste"] == True,
     trechos["taxa_teste"] > p0_teste * 100],
    ["não crítico", "não avaliável", "continua crítico", "acima da média"],
    default="voltou à média")
print(trechos["status_teste"].value_counts())

trechos.to_csv("trechos.csv", index=False, sep=";", encoding="utf-8-sig")

Críticos: 132 | avaliáveis no teste: 118 | não avaliáveis: 14
Críticos: 49.1% (definição) -> 41.7% (teste)
Não críticos: 25.2% (definição) -> 25.5% (teste)
Média nacional: 28.4% -> 28.2%
Parte do excesso de gravidade que se manteve: 65%
Críticos com taxa acima da média nacional no teste: 93%
Críticos que continuam críticos no teste: 10%
Não críticos que viraram críticos no teste: 0%
Correlação de Spearman entre as taxas dos dois períodos: 0.53
status_teste
não crítico         6374
acima da média        98
não avaliável         14
continua crítico      12
voltou à média         8
Name: count, dtype: int64


In [17]:
# --- 17. Análise de sensibilidade ---
from scipy import stats
import numpy as np

df["fatal"] = (df["mortos"] > 0).astype(int)   # versão "só mortes"

def rodar_cenario(nome, min_n, min_n_teste, col_grave):
    d = df.assign(g=df[col_grave])
    base = (d.groupby(["trecho", "periodo"])
              .agg(n=("id", "size"), graves=("g", "sum"))
              .unstack("periodo", fill_value=0))
    base.columns = [f"{a}_{b}" for a, b in base.columns]

    p_def = d.loc[d["periodo"] == "definicao", "g"].mean() * 100
    p_teste = d.loc[d["periodo"] == "teste", "g"].mean() * 100

    # Quem é crítico (definição 2022-2024)
    el = base[base["n_definicao"] >= min_n].copy()
    el["p"] = stats.binom.sf(el["graves_definicao"] - 1, el["n_definicao"], p_def / 100)
    el["q"] = stats.false_discovery_control(el["p"], method="bh")
    el["critico"] = el["q"] < 0.05

    # Quem continua crítico (teste 2025-2026)
    av = el[el["n_teste"] >= min_n_teste].copy()
    av["p_t"] = stats.binom.sf(av["graves_teste"] - 1, av["n_teste"], p_teste / 100)
    av["q_t"] = stats.false_discovery_control(av["p_t"], method="bh")
    av["critico_teste"] = av["q_t"] < 0.05
    crit = av[av["critico"]]

    r = {"cenario": nome,
         "trechos_elegiveis": len(el),
         "trechos_criticos": int(el["critico"].sum()),
         "criticos_avaliaveis": len(crit)}
    if len(crit) > 0:
        t_def = crit["graves_definicao"].sum() / crit["n_definicao"].sum() * 100
        t_teste = crit["graves_teste"].sum() / crit["n_teste"].sum() * 100
        r["taxa_criticos_def_%"] = round(t_def, 1)
        r["taxa_criticos_teste_%"] = round(t_teste, 1)
        r["media_nacional_teste_%"] = round(p_teste, 1)
        r["excesso_mantido_%"] = round((t_teste - p_teste) / (t_def - p_def) * 100)
        r["acima_da_media_%"] = round((crit["graves_teste"] / crit["n_teste"] * 100 > p_teste).mean() * 100)
        r["continuam_criticos_%"] = round(crit["critico_teste"].mean() * 100)
    rho, _ = stats.spearmanr(av["graves_definicao"] / av["n_definicao"],
                             av["graves_teste"] / av["n_teste"])
    r["spearman"] = round(rho, 2)
    return r, set(el.index[el["critico"]]), set(el.index)

cenarios = [("Atual (n>=20)", 20, 11, "grave"),
            ("Corte menor (n>=10)", 10, 6, "grave"),
            ("Corte maior (n>=30)", 30, 17, "grave"),
            ("So mortes (n>=20)", 20, 11, "fatal")]

linhas, ref_crit = [], None
for nome, mn, mt, col in cenarios:
    r, crit_set, elig_set = rodar_cenario(nome, mn, mt, col)
    if ref_crit is None:
        ref_crit = crit_set                      # críticos do cenário atual
    comum = ref_crit & elig_set                  # os que "cabem" neste cenário
    r["criticos_do_atual_mantidos_%"] = round(len(ref_crit & crit_set) / len(comum) * 100) if comum else np.nan
    linhas.append(r)

sens = pd.DataFrame(linhas)
print(sens.set_index("cenario").T.to_string())
sens.to_csv("sensibilidade.csv", index=False, sep=";", encoding="utf-8-sig")

cenario                       Atual (n>=20)  Corte menor (n>=10)  Corte maior (n>=30)  So mortes (n>=20)
trechos_elegiveis                   2229.00              3519.00              1594.00            2229.00
trechos_criticos                     132.00               207.00                96.00              29.00
criticos_avaliaveis                  118.00               183.00                85.00              23.00
taxa_criticos_def_%                   49.10                51.70                47.40              29.10
taxa_criticos_teste_%                 41.70                43.80                40.20              20.60
media_nacional_teste_%                28.20                28.20                28.20               7.20
excesso_mantido_%                     65.00                67.00                63.00              61.00
acima_da_media_%                      93.00                93.00                93.00              96.00
continuam_criticos_%                  10.00            

In [18]:
# --- 18. Exportar tudo para o Power BI ---
def exportar(tabela, nome):
    tabela.to_csv(nome, index=False, sep=";", decimal=",", encoding="utf-8-sig")

# Tabela por trecho (True/False vira Sim/Não, mais fácil de filtrar)
t = trechos.copy()
for c in t.columns:
    if t[c].dtype == bool:
        t[c] = t[c].map({True: "Sim", False: "Não"})
exportar(t, "pbi_trechos.csv")

# Tabela de acidentes (uma linha por acidente), só com as colunas úteis
colunas = ["id", "data_inversa", "ano", "periodo", "trecho", "br", "uf", "faixa_km", "municipio",
           "causa_acidente", "tipo_acidente", "classificacao_acidente", "fase_dia", "dia_semana",
           "condicao_metereologica", "tipo_pista", "tracado_via", "mortos", "feridos_graves",
           "feridos_leves", "veiculos", "grave", "latitude", "longitude"]
exportar(df[colunas], "pbi_acidentes.csv")

exportar(top5_geral, "pbi_top5_causas_geral.csv")
exportar(top5_ano, "pbi_top5_causas_por_ano.csv")
exportar(sens, "pbi_sensibilidade.csv")

In [20]:
import json
import numpy as np

cols = ["trecho","br","uf","faixa_km","lat","lon",
        "n_definicao","graves_definicao","n_teste","graves_teste","causa_principal"]

t = trechos.loc[trechos["n_definicao"] >= 10, cols].copy()
t = t.astype(object).where(t.notna(), None)

anual = (df.groupby("ano")
           .agg(n=("id", "size"), graves=("grave", "sum"))
           .reset_index())

saida = {
    "p0_definicao": float(df.loc[df["periodo"] == "definicao", "grave"].mean() * 100),
    "p0_teste": float(df.loc[df["periodo"] == "teste", "grave"].mean() * 100),
    "min_n_teste": 11,
    "trechos": t.to_dict("records"),
    "anual": anual.to_dict("records"),
    "sensibilidade": sens.to_dict("records"),
    "referencia": {"criticos": int(((trechos["critico"] == True) | (trechos["critico"] == "Sim")).sum())},
}

def conv(o):
    if isinstance(o, (np.integer, np.floating, np.bool_)):
        return o.item()
    return str(o)

with open("dados_monitor.json", "w", encoding="utf-8") as f:
    json.dump(saida, f, ensure_ascii=False, default=conv)

print("OK:", len(t), "trechos | críticos:", saida["referencia"]["criticos"],
      "| p0 def/teste:", round(saida["p0_definicao"], 1), round(saida["p0_teste"], 1))

OK: 3519 trechos | críticos: 132 | p0 def/teste: 28.4 28.2
